### promptTemplate
- 原始方式：字符串拼接（f-string、+）
    - 缺点：当变量多、或者需要构建多轮对话（系统消息+用户消息+AI消息）时，代码会变得极其混乱、极易拼写错误，且难以维护。
- ChatPromptTemplate
    - 优点：结构清晰：系统消息（system）和用户消息（human）泾渭分明。
            自动校验：会自动校验：如果漏填了 {topic}，LangChain 会直接报错，而不是让模型收到错误的提示词。
    - 便于复用：可以把这个模板存起来，在不同地方调用。
```python
from langchain_core.prompts import ChatPromptTemplate

# 1. 定义模板结构（用占位符 {变量} 代替具体内容）
chat_template = ChatPromptTemplate.from_messages([#也可直接实例化构建ChatPromptTemplate(...)
    ("system", "你是一个{difficulty}级别的编程导师。"),
    ("human", "请用简单易懂的语言解释{topic}。")
])

# 2. 实例化：传入具体变量（LangChain 会自动校验变量是否齐全）
prompt_value = chat_template.invoke({"difficulty": "初学者", "topic": "Python"})
```
- 3种调用方式
    - 方式1：使用 invoke() （返回 ChatPromptValue）
    - 方式2：使用 format() （返回字符串）不推荐
    - 方式3：使用 format_messages() （返回消息列表）自己手动解析时
- 6种传参方式
    - 类型1：str 列表类型
        - ChatPromptTemplate("你好")默认为human消息
    - 类型2：tuple 列表类型
        - ChatPromptTemplate(("system", "你好是一个{difficulty}级别的编程导师。"), ("human", "请用简单易懂的语言解释{topic}。"))
    - 类型3：dict 列表类型
        - ChatPromptTemplate.from_messages([("system", "你好是一个{difficulty}级别的编程导师。"), ("human", "请用简单易懂的语言解释{topic}。")])
    - 类型4：Message 列表类型
        - ChatPromptTemplate.from_messages([SystemMessage(content="你好是一个{difficulty}级别的编程导师。"), HumanMessage(content="请用简单易懂的语言解释{topic}。")])
    - 类型5：MessagePromptTemplate 列表类型
        - ChatPromptTemplate.from_messages([MessagePromptTemplate.from_messages([("system", "你好是一个{difficulty}级别的编程导师。"), ("human", "请用简单易懂的语言解释{topic}。")])])
    - 类型6：BaseChatPromptTemplate 列表类型
        - ChatPromptTemplate.from_messages([BaseChatPromptTemplate.from_messages([("system", "你好是一个{difficulty}级别的编程导师。"), ("human", "请用简单易懂的语言解释{topic}。")])])

### 高级特性
- 部分变量预填充：partial()
```python
from langchain_core.prompts import ChatPromptTemplate
# 原始模板
template = ChatPromptTemplate.from_messages([
("system", "你是{role}，目标用户是{audience}"),
("user", "{task}")
])
# 部分填充
customer_support_template = template.partial(
role="客服专员",
audience="普通用户"
)
# 现在只需要提供 task
messages = customer_support_template.invoke({"task":"解释退款政策"})
print(messages)
```
- 消息占位符
    - 方式1：JSON 形式
    - 方式2：MessagesPlaceholder 实例
        - 例如：MessagesPlaceholder(variable_name="history")
```python
from langchain_core.prompts import ChatPromptTemplate
template = ChatPromptTemplate.from_messages([
    ("system", "你是一个有用的AI助手"),
    ("placeholder", "{conversation}"),#JSON格式
    MessagesPlaceholder(variable_name="history"),#实例格式
])
prompt_value = template.invoke({
    "conversation": [
        ("human", "你好!"),
        ("ai", "今天我能帮你做什么？"),
        ("human", "你能给我做一个冰激凌吗？"),
        ("ai", "抱歉，我没有这样的能力"),
    ],
    "history": [
        ("human", "我叫XXX"),
        ("ai", "你好，XXX！")
    ]
})
```
- 可复用模板库(详见文档)
    - 举例1：在 templates.py 中统一声明模板
    - 举例2：按模块（翻译、编程等）拆分管理模板
- 模板组合(详见文档)
    - 方法1：字符串组合
    - 方法2：使用 + 运算符组合模板